# Transformer realignment: results

Reads every `models/Transformer/*/experiments/*/metrics.json` (picking each
experiment's most recent run) plus `models/comparison/lstm_on_shared_test_split/metrics.json`
(Larry's LSTM scored on the same 16-city, 72h->24h test windows), and tells
the ablation story: does temperature history help, does city identity help,
do they combine, and does residual/baseline blending close the gap to the
LSTM's near-zero bias.

Run this after `run_all_experiments.py` has produced all five experiments'
`metrics.json` files. Each cell below is self-contained given the loaded
data, so partial results (e.g. only exp03 and exp00 done) still render --
missing experiments are simply absent from the tables/plots, not an error.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd().parent

TRANSFORMER_DIR = PROJECT_ROOT / "models" / "Transformer"
LSTM_COMPARISON_FILE = PROJECT_ROOT / "models" / "comparison" / "lstm_on_shared_test_split" / "metrics.json"

EXPERIMENT_ORDER = [
    "exp00_baseline",
    "exp01_temperature",
    "exp02_city",
    "exp03_temperature_city",
    "exp04_baseline_blend",
]

In [ ]:
def latest_experiment_dir(experiment_name):
    experiments_dir = TRANSFORMER_DIR / experiment_name / "experiments"
    if not experiments_dir.exists():
        return None
    candidates = sorted((d for d in experiments_dir.iterdir() if d.is_dir()), reverse=True)
    for candidate in candidates:
        if (candidate / "metrics.json").exists():
            return candidate
    return None


def total_params(experiment_dir):
    summary_file = experiment_dir / "model_summary.txt"
    if not summary_file.exists():
        return None
    text = summary_file.read_text()
    for line in text.splitlines():
        if line.strip().startswith("Total params:"):
            digits = "".join(ch for ch in line.split(":", 1)[1] if ch.isdigit())
            return int(digits) if digits else None
    return None


def epochs_trained(experiment_dir):
    history_file = experiment_dir / "training_history.csv"
    if not history_file.exists():
        return None
    return len(pd.read_csv(history_file))


def is_city_aware(experiment_dir):
    # city_aware isn't a TransformerConfig field (it's a run_training()-level
    # switch, like feature_columns), so it isn't in config.json -- infer it
    # from whether the saved model has a city embedding layer instead.
    summary_file = experiment_dir / "model_summary.txt"
    if not summary_file.exists():
        return None
    return "city_id" in summary_file.read_text()


results = {}
for experiment_name in EXPERIMENT_ORDER:
    experiment_dir = latest_experiment_dir(experiment_name)
    if experiment_dir is None:
        print(f"{experiment_name}: no metrics.json yet -- skipping")
        continue
    with open(experiment_dir / "metrics.json") as f:
        metrics = json.load(f)
    with open(experiment_dir / "config.json") as f:
        config = json.load(f)
    results[experiment_name] = {
        "experiment_dir": experiment_dir,
        "metrics": metrics,
        "config": config,
        "params": total_params(experiment_dir),
        "epochs_trained": epochs_trained(experiment_dir),
    }

lstm_metrics = None
if LSTM_COMPARISON_FILE.exists():
    with open(LSTM_COMPARISON_FILE) as f:
        lstm_metrics = json.load(f)

print(f"Loaded {len(results)}/{len(EXPERIMENT_ORDER)} Transformer experiments.")
print(f"LSTM comparison metrics {'found' if lstm_metrics else 'NOT found'} at {LSTM_COMPARISON_FILE}")

## Summary table

Config, feature count, parameter count, epochs trained, overall MAE/RMSE/bias, and a skill score against persistence (`1 - model_mae / persistence_mae`, as a percentage -- how much of the persistence baseline's error the model removed).

In [ ]:
def skill_score(model_mae, persistence_mae):
    return 100.0 * (1.0 - model_mae / persistence_mae)


rows = []
for experiment_name in EXPERIMENT_ORDER:
    if experiment_name not in results:
        continue
    entry = results[experiment_name]
    metrics, config = entry["metrics"], entry["config"]
    model_metrics = metrics["improved_model"]
    persistence_mae = metrics["persistence_baseline"]["overall_mae"]

    rows.append({
        "experiment": experiment_name,
        "features": len(metrics.get("feature_columns", [])),
        "city_aware": is_city_aware(entry["experiment_dir"]),
        "baseline_blend": config.get("baseline_blend", False),
        "params": entry["params"],
        "epochs": entry["epochs_trained"],
        "overall_mae": model_metrics["overall_mae"],
        "overall_rmse": model_metrics["overall_rmse"],
        "overall_bias": model_metrics["overall_bias"],
        "persistence_mae": persistence_mae,
        "skill_vs_persistence_pct": skill_score(model_metrics["overall_mae"], persistence_mae),
    })

summary_columns = ["experiment", "features", "city_aware", "baseline_blend", "params", "epochs",
                   "overall_mae", "overall_rmse", "overall_bias", "persistence_mae",
                   "skill_vs_persistence_pct"]
summary_df = pd.DataFrame(rows, columns=summary_columns).set_index("experiment")
summary_df

## MAE by forecast horizon: all five configs plus both baselines

Persistence and previous-day are built from the target's own raw history (`temperature_history`), which doesn't depend on `feature_columns` -- so they're numerically identical across all five experiments. The plot below takes them from whichever experiment loaded first.

In [ ]:
plt.figure(figsize=(12, 7))
hours = np.arange(1, 25)

baseline_source = next(iter(results.values()), None)
if baseline_source is not None:
    plt.plot(hours, baseline_source["metrics"]["persistence_baseline"]["mae_by_hour"],
              linestyle="--", color="gray", label="Persistence")
    plt.plot(hours, baseline_source["metrics"]["previous_day_baseline"]["mae_by_hour"],
              linestyle=":", color="gray", label="Previous day")

for experiment_name in EXPERIMENT_ORDER:
    if experiment_name not in results:
        continue
    mae_by_hour = results[experiment_name]["metrics"]["improved_model"]["mae_by_hour"]
    plt.plot(hours, mae_by_hour, marker="o", markersize=3, label=experiment_name)

if lstm_metrics is not None:
    plt.plot(hours, lstm_metrics["improved_model"]["mae_by_hour"],
              marker="s", markersize=3, linewidth=2, color="black", label="LSTM (Larry, shared test split)")

plt.title("MAE by Forecast Horizon: Transformer Ablation vs. Baselines vs. LSTM")
plt.xlabel("Forecast Hour")
plt.ylabel("MAE (°C)")
plt.xticks(hours)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Per-experiment narrative

### exp00: baseline (control)

**Hypothesis**: not hypothesis-driven -- the control every other experiment is measured against.

In [ ]:
if "exp00_baseline" in results:
    m = results["exp00_baseline"]["metrics"]["improved_model"]
    p = results["exp00_baseline"]["metrics"]["persistence_baseline"]["overall_mae"]
    print(f"exp00_baseline: overall MAE {m['overall_mae']:.3f} degC, RMSE {m['overall_rmse']:.3f}, "
          f"bias {m['overall_bias']:+.3f}. Beats persistence ({p:.3f}) by "
          f"{skill_score(m['overall_mae'], p):.1f}%.")
else:
    print("exp00_baseline not trained yet.")

### exp01: temperature as an input feature

**Hypothesis**: does the target's own recent history improve on exp00's weather-only inputs, and does the effect hold or decay at longer horizons?

In [ ]:
if {"exp00_baseline", "exp01_temperature"} <= results.keys():
    base = results["exp00_baseline"]["metrics"]["improved_model"]
    exp1 = results["exp01_temperature"]["metrics"]["improved_model"]
    delta = base["overall_mae"] - exp1["overall_mae"]
    direction = "improved" if delta > 0 else "worsened"
    print(f"exp01_temperature vs exp00_baseline: overall MAE {exp1['overall_mae']:.3f} vs "
          f"{base['overall_mae']:.3f} ({direction} by {abs(delta):.3f} degC).")

    early = np.mean(exp1["mae_by_hour"][:6]) - np.mean(base["mae_by_hour"][:6])
    late = np.mean(exp1["mae_by_hour"][-6:]) - np.mean(base["mae_by_hour"][-6:])
    print(f"MAE delta (exp01 - exp00), h1-6 avg: {early:+.3f} degC; h19-24 avg: {late:+.3f} degC "
          f"-- {'the temperature-history effect decays with horizon' if abs(early) > abs(late) else 'the effect holds across the forecast'}.")
else:
    print("exp00_baseline and/or exp01_temperature not trained yet.")

### exp02: city identity as an input feature

**Hypothesis**: does a learned city embedding help exp00's weather-only baseline, and is the effect larger at longer horizons (where the model has less to lean on from recent history and more to gain from local climatology)?

In [ ]:
if {"exp00_baseline", "exp02_city"} <= results.keys():
    base = results["exp00_baseline"]["metrics"]["improved_model"]
    exp2 = results["exp02_city"]["metrics"]["improved_model"]
    delta = base["overall_mae"] - exp2["overall_mae"]
    direction = "improved" if delta > 0 else "worsened"
    print(f"exp02_city vs exp00_baseline: overall MAE {exp2['overall_mae']:.3f} vs "
          f"{base['overall_mae']:.3f} ({direction} by {abs(delta):.3f} degC).")

    early = np.mean(exp2["mae_by_hour"][:6]) - np.mean(base["mae_by_hour"][:6])
    late = np.mean(exp2["mae_by_hour"][-6:]) - np.mean(base["mae_by_hour"][-6:])
    print(f"MAE delta (exp02 - exp00), h1-6 avg: {early:+.3f} degC; h19-24 avg: {late:+.3f} degC "
          f"-- {'city identity matters more at long horizons, as hypothesized' if late < early else 'no clear long-horizon advantage from city identity'}.")
else:
    print("exp00_baseline and/or exp02_city not trained yet.")

### exp03: combining temperature history and city identity

**Hypothesis**: are exp01's and exp02's effects additive (exp03 at least as good as the better of the two alone) or interactive?

In [ ]:
needed = {"exp00_baseline", "exp01_temperature", "exp02_city", "exp03_temperature_city"}
if needed <= results.keys():
    base = results["exp00_baseline"]["metrics"]["improved_model"]["overall_mae"]
    exp1 = results["exp01_temperature"]["metrics"]["improved_model"]["overall_mae"]
    exp2 = results["exp02_city"]["metrics"]["improved_model"]["overall_mae"]
    exp3 = results["exp03_temperature_city"]["metrics"]["improved_model"]["overall_mae"]

    better_alone = min(exp1, exp2)
    print(f"overall MAE -- exp00: {base:.3f}, exp01: {exp1:.3f}, exp02: {exp2:.3f}, exp03: {exp3:.3f}")
    if exp3 <= better_alone:
        print(f"exp03 ({exp3:.3f}) beats the better of exp01/exp02 ({better_alone:.3f}) -- "
              "the effects are additive (or better).")
    else:
        print(f"exp03 ({exp3:.3f}) is worse than the better of exp01/exp02 ({better_alone:.3f}) -- "
              "the effects interact negatively when combined.")
else:
    print("Not all of exp00/exp01/exp02/exp03 are trained yet.")

### exp04: residual/baseline blending

**Hypothesis**: Larry's LSTM predicts a correction to a persistence-like estimate rather than the raw temperature, with an h=3 bias of +0.07 degC against the pre-realignment Transformer's -1.67 degC. Does giving the Transformer the same residual formulation close the bias gap?

In [ ]:
if {"exp03_temperature_city", "exp04_baseline_blend"} <= results.keys():
    exp3 = results["exp03_temperature_city"]["metrics"]["improved_model"]
    exp4 = results["exp04_baseline_blend"]["metrics"]["improved_model"]

    print(f"overall MAE  -- exp03: {exp3['overall_mae']:.3f}, exp04: {exp4['overall_mae']:.3f}")
    print(f"overall bias -- exp03: {exp3['overall_bias']:+.3f}, exp04: {exp4['overall_bias']:+.3f}")

    if lstm_metrics is not None:
        lstm_bias = lstm_metrics["improved_model"]["overall_bias"]
        print(f"LSTM overall bias: {lstm_bias:+.3f}")
        moved_toward_lstm = abs(exp4["overall_bias"] - lstm_bias) < abs(exp3["overall_bias"] - lstm_bias)
        print("baseline_blend moved bias toward the LSTM's" if moved_toward_lstm
              else "baseline_blend did not move bias closer to the LSTM's")
else:
    print("exp03_temperature_city and/or exp04_baseline_blend not trained yet.")

## Per-city breakdown for the best config

"Best" = lowest `overall_mae` among trained experiments.

In [ ]:
if results:
    best_name = min(results, key=lambda name: results[name]["metrics"]["improved_model"]["overall_mae"])
    best_metrics = results[best_name]["metrics"]
    print(f"Best config so far: {best_name} (overall MAE {best_metrics['improved_model']['overall_mae']:.3f})")

    by_group = best_metrics.get("test_metrics_by_group")
    if by_group:
        city_df = pd.DataFrame({
            city: {"overall_mae": m["overall_mae"], "overall_bias": m["overall_bias"], "samples": m["samples"]}
            for city, m in by_group.items()
        }).T.sort_values("overall_mae")

        plt.figure(figsize=(10, 6))
        plt.barh(city_df.index, city_df["overall_mae"])
        plt.xlabel("MAE (°C)")
        plt.title(f"Per-City MAE -- {best_name}")
        plt.gca().invert_yaxis()
        plt.grid(True, axis="x")
        plt.tight_layout()
        plt.show()

        city_df
    else:
        print(f"{best_name}'s metrics.json has no test_metrics_by_group.")
else:
    print("No experiments trained yet.")

## Final comparison panel: best Transformer config vs. Larry's LSTM

Both are scored on the identical shared 16-city, 72h->24h test windows with
identical metric code (`src.models.common.evaluation.calculate_metrics`),
so city-set size and architecture are the only stated differences.

In [ ]:
if results and lstm_metrics is not None:
    best_name = min(results, key=lambda name: results[name]["metrics"]["improved_model"]["overall_mae"])
    best = results[best_name]["metrics"]["improved_model"]
    lstm = lstm_metrics["improved_model"]

    comparison_df = pd.DataFrame({
        best_name: {"overall_mae": best["overall_mae"], "overall_rmse": best["overall_rmse"], "overall_bias": best["overall_bias"]},
        "LSTM (Larry)": {"overall_mae": lstm["overall_mae"], "overall_rmse": lstm["overall_rmse"], "overall_bias": lstm["overall_bias"]},
    }).T
    display(comparison_df)

    plt.figure(figsize=(10, 6))
    plt.plot(hours, best["mae_by_hour"], marker="o", label=best_name)
    plt.plot(hours, lstm["mae_by_hour"], marker="s", label="LSTM (Larry)")
    plt.title(f"{best_name} vs. LSTM: MAE by Forecast Horizon")
    plt.xlabel("Forecast Hour")
    plt.ylabel("MAE (°C)")
    plt.xticks(hours)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    gap_pct = 100.0 * (best["overall_mae"] - lstm["overall_mae"]) / lstm["overall_mae"]
    print(f"Best Transformer config's overall MAE is {gap_pct:+.1f}% relative to the LSTM's "
          f"({'worse' if gap_pct > 0 else 'better'}). The LSTM trains on ~84 cities' worth of history "
          f"and a residual/baseline-blend architecture from the start; {best_name} trains on 16.")
else:
    print("Need at least one trained Transformer experiment and the LSTM comparison metrics.json.")